In [ ]:
## Restructured model with germany data

In [21]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)['contacts']

In [2]:
from cntmosaic.dataloader.restru_loaders import MergedLoader, RawLoader
import pandas as pd
import numpy as np
import xarray as xr
from cntmosaic.dataloader import CoordToColumns
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [ ]:
#c = pd.read_csv('2008_Mossong_POLYMOD_contact_common.csv')
#p = pd.read_csv('2008_Mossong_POLYMOD_participant_common.csv')
#mapp = CoordToColumns(age_part='part_age', age_cnt='cnt_age_exact', id_var='part_id')
#data = RawLoader(p, c, mapp)

In [22]:
mapp = CoordToColumns(age_part='age_part', age_cnt='age_cnt', id_var='id')

In [23]:
d = MergedLoader(data, mapp)

In [24]:
d.load()

In [25]:
model = restr_BRCfine(d)

new model instantiated, please check default hyperparameters


In [35]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data1 = pickle.load(file)['population']
data1 = data1[data1.age < 85]
data1 = data1.groupby('age')['P'].sum()
age_dist = data1/data1.sum()

In [36]:
model.params.age_dist = age_dist.values

In [37]:
model.compile()

model compiled, ready for sampling


In [38]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 1000,
    num_warmup = 500,
    num_chains = 2)

/home/yiminglin/Downloads/cntmosaic/cntmosaic/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 2 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(2)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1500/1500 [47:47<00:00,  1.91s/it, 511 steps of size 1.19e-02. acc. prob=0.94]  


Number of divergences: 0


In [39]:
# Save to a file
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)


In [3]:
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)


In [40]:
from cntmosaic.analysis import ModelSummariserMCMC, ModelVisualiser

In [41]:
MSM = ModelSummariserMCMC(model)

In [42]:
visual = ModelVisualiser(MSM)

In [43]:
visual.plot_cint()

alt.Chart(...)